# Adım 4 — Hava durumu ekleme ve adil karşılaştırma

**Amaç:** Aynı London Smart Meters eğitim/test hanelerinde Random Forest modelini *hava durumu olmadan* ve *önceki günün bilinen hava durumu bilgileriyle* karşılaştırmak.

Gerçekleşmiş **test günü** hava durumunu kullanmıyoruz; bu, ertesi gün tahmininde bilgi sızıntısı olur. Sonuçlar veri indirilip deney çalıştırılmadan iddia edilemez.

Kaynaklar: London Smart Meters alt kümesi (Hugging Face tekrar yayını); tarihsel Dark Sky verisinin 30 dakikalık yeniden işlenmiş GitHub kopyası.

## 1 — London elektrik tüketim alt kümesini indir

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","huggingface-hub"])
from huggingface_hub import hf_hub_download
from pathlib import Path
REPO="Sufiyan83/Low-Carbon-London-Smart-Meter-Cleaned-FeatureReady"
FILE="Partitioned LCL Data/Small LCL Data/LCL-June2015v2_0.csv"
source_file=Path(hf_hub_download(repo_id=REPO,filename=FILE,repo_type="dataset"))
print(f"✅ Tuketim verisi hazir: {source_file.name}, boyut: {source_file.stat().st_size/1e6:.1f} MB")

## 2 — Gerçek tarihsel London hava durumu dosyasını indir

GitHub kopyası. Dosya doğrudan indirilir; tarayıcıda açmanız gerekmez. İndirme başarısız olursa çıktıyı paylaşın.

In [ ]:
import requests
import pandas as pd
WEATHER_URL="https://raw.githubusercontent.com/Iavalos/Forecasting-electricity-consumption-in-London/master/weather_hourly_darksky_30minutes.csv"
weather_file=Path("weather_hourly_darksky_30minutes.csv")
resp=requests.get(WEATHER_URL,timeout=120)
resp.raise_for_status()
weather_file.write_bytes(resp.content)
if weather_file.stat().st_size < 100_000:
    raise ValueError("Hava dosyasi beklenmedik sekilde kucuk: kaynak dogrulanmali")
print("✅ Hava verisi indirildi | MB:",round(weather_file.stat().st_size/1e6,2))
print("Sutunlar:",pd.read_csv(weather_file,nrows=3).columns.tolist())

## 3 — Eğitim, test ve analiz fonksiyonları

In [ ]:
"""London Smart Meters Step 4: fair day-ahead weather feature ablation.
Requires source London CSV and real London historical weather CSV.
Weather features are shifted by 24h (known at 00:00 next day); never use same-day actuals.
"""
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
MAX_HOMES = 12
MIN_HOURS = 24 * 75
METEO_NAMES = ["temperature", "apparentTemperature", "humidity", "windSpeed", "pressure", "cloudCover"]


def parse_datetime(x):
    s = x.astype(str).str.strip()
    numeric = pd.to_numeric(s, errors="coerce")
    if numeric.notna().mean() > .95:
        median = numeric.dropna().abs().median()
        unit = "ms" if median > 1e11 else "s"
        return pd.to_datetime(numeric, unit=unit, utc=True, errors="coerce").dt.tz_localize(None)
    # Mixed str formats with timezone strings, UTC conversion for consistent alignment.
    return pd.to_datetime(s, format="mixed", utc=True, errors="coerce").dt.tz_localize(None)


def get_meteo(path):
    raw = pd.read_csv(path, low_memory=False)
    raw.columns = raw.columns.astype(str).str.strip()
    lookup = {col.lower():col for col in raw.columns}
    tcol = next((lookup[x] for x in ["time", "datetime", "date", "timestamp"] if x in lookup), None)
    if tcol is None:
        raise ValueError(f"Hava dosyasinda zaman sutunu bulunamadi: {list(raw.columns)}")
    usable = [c for c in METEO_NAMES if c.lower() in lookup]
    if not usable:
        raise ValueError(f"Hava dosyasinda sayisal hava degiskeni bulunamadi: {list(raw.columns)}")
    frame = pd.DataFrame({"hour":parse_datetime(raw[tcol]).dt.floor("h")})
    for feature in usable:
        frame[feature] = pd.to_numeric(raw[lookup[feature.lower()]], errors="coerce")
    frame = frame.dropna(subset=["hour"])
    # 30-minute interpolated file => two values/hour: take average. No future data used.
    weather_hourly = frame.groupby("hour",as_index=True)[usable].mean().sort_index()
    weather_hourly = weather_hourly.reindex(pd.date_range(weather_hourly.index.min(), weather_hourly.index.max(), freq="h"))
    # 24-hour lag = weather at same hour yesterday, known at midnight ahead of forecast day.
    weather_hourly = weather_hourly.shift(24)
    weather_cols = [f"yesterday_{x}" for x in usable]
    weather_hourly.columns = weather_cols
    result = weather_hourly.rename_axis("hour").reset_index()
    return result,weather_cols,usable


def get_smart_meters(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    lower = {c.lower():c for c in df.columns}
    idcol = lower.get("lclid")
    timecol = lower.get("datetime")
    kwhcol = next((c for c in df.columns if "kwh" in c.lower()), None)
    if not all([idcol,timecol,kwhcol]):
        raise ValueError(f"Smart-meter CSV eksik sutun: {df.columns.tolist()}")
    dt = pd.to_datetime(df[timecol].astype(str).str[:19], errors="coerce")
    val = pd.to_numeric(df[kwhcol].replace("Null", np.nan), errors="coerce")
    base = pd.DataFrame({"home":df[idcol].astype(str),"time":dt,"half_kwh":val})
    base = base.dropna(subset=["home","time","half_kwh"])
    base = base[(base.half_kwh >= 0) & (base.half_kwh <= 30)]
    # Half-hour timestamp marks period end.
    base["hour"] = (base.time - pd.Timedelta(minutes=30)).dt.floor("h")
    hourly = (base.groupby(["home","hour"], as_index=False)
              .agg(kwh=("half_kwh","sum"),slots=("time","nunique")))
    hourly = hourly[hourly.slots == 2].drop(columns="slots")
    counts = hourly.groupby("home").size().sort_values(ascending=False)
    homes = counts[counts >= MIN_HOURS].head(MAX_HOMES).index.tolist()
    if len(homes) < 4: raise ValueError("Yeterli hane yok; en az 4 hane, 75 gun gerekir.")
    hourly = hourly[hourly.home.isin(homes)]
    frames=[]
    for home,group in hourly.groupby("home"):
        s=group.set_index("hour").kwh.sort_index()
        s=s[~s.index.duplicated(keep="first")]
        s=s.reindex(pd.date_range(s.index.min(),s.index.max(),freq="h"))
        frames.append(pd.DataFrame({"home":home,"hour":s.index,"kwh":s.to_numpy()}))
    data=pd.concat(frames,ignore_index=True).sort_values(["home","hour"])
    g=data.groupby("home",sort=False)["kwh"]
    for lag in [24,48,168]: data[f"lag_{lag}"]=g.shift(lag)
    data["past_24h_mean"]=data.groupby("home",sort=False)["kwh"].transform(lambda s:s.shift(24).rolling(24,min_periods=24).mean())
    data["past_7d_mean"]=data.groupby("home",sort=False)["kwh"].transform(lambda s:s.shift(24).rolling(168,min_periods=168).mean())
    data["tod"]=data.hour.dt.hour
    data["weekday"]=data.hour.dt.dayofweek
    data["month"]=data.hour.dt.month
    data["weekend"]=(data.weekday>=5).astype(int)
    data["hour_sin"]=np.sin(2*np.pi*data.tod/24)
    data["hour_cos"]=np.cos(2*np.pi*data.tod/24)
    return data,homes


def run(meter_csv, weather_csv, outdir="HavaDurumu_Sonuclar"):
    out=Path(outdir);out.mkdir(parents=True,exist_ok=True)
    data,homes=get_smart_meters(meter_csv)
    weather, weather_cols, original_names=get_meteo(weather_csv)
    all_features=["lag_24","lag_48","lag_168","past_24h_mean","past_7d_mean","tod","weekday","month","weekend","hour_sin","hour_cos"]
    before=len(data.dropna(subset=all_features+["kwh"]))
    joined=data.merge(weather,on="hour",how="left",validate="many_to_one")
    clean=joined.dropna(subset=all_features+weather_cols+["kwh"]).copy()
    if len(clean) < 5000:
        raise ValueError(f"Hava ve sayac saatleri az eslesti: {len(clean)}. Tarih/saat formatlari kontrol edilmeli.")
    r=np.random.default_rng(SEED)
    shuffled=r.permutation(homes)
    n_heldout=max(1,int(np.ceil(len(homes)*.25)))
    heldout=sorted(shuffled[:n_heldout].tolist()); train_homes=sorted(shuffled[n_heldout:].tolist())
    # Use the same 80%-chronological cutoff from the prior no-weather experiment.
    # Its candidate dates come from meter observations only, not weather availability.
    dates=pd.DatetimeIndex(data.dropna(subset=all_features+["kwh"]).hour.dt.floor("D").unique()).sort_values()
    cutoff=dates[int(len(dates)*.8)]
    train=clean[(clean.home.isin(train_homes))&(clean.hour<cutoff)].copy()
    test=clean[(clean.home.isin(heldout))&(clean.hour>=cutoff)].copy()
    if len(train)<200 or len(test)<100: raise ValueError(f"Az eslesen train/test: {len(train)}, {len(test)}")
    assert set(train.home).isdisjoint(test.home)
    assert train.hour.max()<test.hour.min()
    preds={}
    for key, features in [("RF_without_weather",all_features),("RF_with_yesterday_weather",all_features+weather_cols)]:
        rf=RandomForestRegressor(n_estimators=80,max_depth=18,min_samples_leaf=3,random_state=SEED,n_jobs=-1)
        rf.fit(train[features],train.kwh)
        preds[key]=rf.predict(test[features]).clip(min=0)
        test[key]=preds[key]
    report=[]
    for key in preds:
        report.append({"model":key,"MAE_kWh":mean_absolute_error(test.kwh,test[key]),"RMSE_kWh":np.sqrt(mean_squared_error(test.kwh,test[key]))})
    metrics=pd.DataFrame(report)
    base_mae=metrics.loc[0,"MAE_kWh"];meteo_mae=metrics.loc[1,"MAE_kWh"]
    improvement=100*(base_mae-meteo_mae)/base_mae
    perhome=[]
    for home,grp in test.groupby("home"):
        perhome.append({"home":home,"n":len(grp),
                        "MAE_without":mean_absolute_error(grp.kwh,grp.RF_without_weather),
                        "MAE_with":mean_absolute_error(grp.kwh,grp.RF_with_yesterday_weather)})
    ph=pd.DataFrame(perhome)
    metrics.to_csv(out/"weather_vs_no_weather_metrics.csv",index=False)
    ph.to_csv(out/"weather_per_home_metrics.csv",index=False)
    test[["home","hour","kwh",*preds]].to_csv(out/"weather_predictions.csv",index=False)
    day=(test[test.home==heldout[0]].copy());day["date"]=day.hour.dt.date
    bestdate=day.groupby("date").size().sort_values(ascending=False).index[0]
    day=day[day.date==bestdate].sort_values("hour")
    fig,ax=plt.subplots(figsize=(11,4.3))
    ax.plot(day.hour.dt.hour,day.kwh,label="Observed",lw=1.9)
    ax.plot(day.hour.dt.hour,day.RF_without_weather,label="RF without weather",lw=1.8)
    ax.plot(day.hour.dt.hour,day.RF_with_yesterday_weather,label="RF + yesterday weather",lw=1.8)
    ax.set(xlabel="Hour",ylabel="kWh per hour",title=f"New household {heldout[0]} / {bestdate}")
    ax.legend();fig.tight_layout();fig.savefig(out/"weather_comparison_example.png",dpi=165);plt.close(fig)
    notes=(
      "Step 4 - historical weather ablation, London\n"
      f"Smart-meter source: {Path(meter_csv).name}\nWeather source: {Path(weather_csv).name}\n"
      f"Training households: {train_homes}\nNever-seen test households: {heldout}\n"
      f"Chronological cutoff: {cutoff}\nTraining rows: {len(train)}; test rows: {len(test)}\n"
      f"Weather variables used (24-hour lag): {original_names}\n"
      f"Rows with historical features before weather merge: {before}; rows after weather merge: {len(clean)}\n"
      "Models evaluated on exactly the same rows; no household ID feature; no realized weather from forecast day.\n"
      f"MAE improvement from weather: {improvement:.3f}% (negative means worse).\n"
      "The weather file is a republished/processed historical London weather file; check provenance.\n"
      "This is a lagged-weather test, NOT a test with next-day weather forecasts.\n"
    )
    (out/"methodology_weather.txt").write_text(notes,encoding="utf8")
    print(notes)
    print(metrics.round(5).to_string(index=False))
    print("Per-household:")
    print(ph.round(5).to_string(index=False))
    return metrics,ph


## 4 — İki modeli aynı gözlemler üzerinde eğit ve karşılaştır

Negatif iyileşme, hava durumu değişkenlerinin hatayı artırdığını gösterir; bu da geçerli bir deney sonucudur.

In [ ]:
from IPython.display import display, Image
metrics, per_home = run(source_file,weather_file)
print("✅ Hava durumu deneyi tamamlandi")
display(metrics)
display(per_home)
display(Image(filename="HavaDurumu_Sonuclar/weather_comparison_example.png"))

## 5 — Sonuçları ZIP olarak indir

In [ ]:
from pathlib import Path
import shutil
from google.colab import files
zip_path=shutil.make_archive("HavaDurumu_Sonuclar","zip","HavaDurumu_Sonuclar")
files.download(zip_path)
print("✅ Sonuc arsivi hazir:",zip_path)